In [2]:
import pandas as pd
import glob
import os
import sqlite3

# Mapeia todos os arquivos .csv dentro da pasta 'data'
# Assumindo que o seu notebook está na mesma pasta raiz que a pasta 'data'
caminho_arquivos = glob.glob(os.path.join('data', '*.csv'))

# Cria uma lista vazia para armazenar os DataFrames individuais
lista_dfs = []

# Loop para ler cada arquivo e adicioná-lo à lista
for arquivo in caminho_arquivos:
    # Lendo o CSV. Ajuste o 'sep' e 'encoding' conforme necessário para o seu dado específico
    df_temp = pd.read_csv(arquivo, sep=';', encoding='utf-8') 
    lista_dfs.append(df_temp)

# Concatena (empilha) todos os DataFrames da lista verticalmente
df_unificado = pd.concat(lista_dfs, ignore_index=True)

# Coleta as dimensões do DataFrame final
total_linhas, total_colunas = df_unificado.shape

print(f"Total de arquivos lidos: {len(caminho_arquivos)}")
print(f"Total de linhas (instâncias na UNF): {total_linhas}")
print(f"Total de colunas: {total_colunas}")

# 1. Conectar ao SQLite (cria o arquivo do banco no seu computador)
conn = sqlite3.connect('trabalho_pratico_2.db')

# 2. Enviar o DataFrame unificado inteiro para o banco como uma "Tabela Raw/Bruta"
# O if_exists='replace' garante que você pode rodar a célula várias vezes sem erro
df_unificado.to_sql('dados_brutos', conn, if_exists='replace', index=False)

Total de arquivos lidos: 7
Total de linhas (instâncias na UNF): 342716
Total de colunas: 25


342716

In [3]:
cursor = conn.cursor()

# 1. Otimização recomendada: Desativar temporariamente checagem de FK e ativar modo síncrono off
cursor.execute("PRAGMA foreign_keys = OFF;")
cursor.execute("PRAGMA synchronous = OFF;")
cursor.execute("PRAGMA journal_mode = MEMORY;")

print("Iniciando a criação do esquema e migração dos dados...")

# ==========================================
# DDL - CRIAÇÃO DAS TABELAS (3FN)
# ==========================================

# Tabela 1: Instituição
cursor.execute('''
CREATE TABLE IF NOT EXISTS Instituicao (
    cnpj_instituicao TEXT PRIMARY KEY,
    nome_instituicao TEXT,
    esfera TEXT,
    municipio_instituicao TEXT,
    uf TEXT
);
''')

# Tabela 2: Fabricante
cursor.execute('''
CREATE TABLE IF NOT EXISTS Fabricante (
    cnpj_fabricante TEXT PRIMARY KEY,
    fabricante TEXT,
    ativo_ano_anterior TEXT,
    qtd_produtos_distintos INTEGER
);
''')

# Tabela 3: Produto
cursor.execute('''
CREATE TABLE IF NOT EXISTS Produto (
    codigo_br TEXT PRIMARY KEY,
    descricao_catmat TEXT,
    generico TEXT,
    anvisa TEXT,
    capacidade TEXT,
    unidade_medida TEXT,
    unidade_fornecimento_capacidade TEXT,
    cnpj_fabricante TEXT,
    FOREIGN KEY (cnpj_fabricante) REFERENCES Fabricante(cnpj_fabricante)
);
''')

# Tabela 4: Fornecedor
cursor.execute('''
CREATE TABLE IF NOT EXISTS Fornecedor (
    cnpj_fornecedor TEXT PRIMARY KEY,
    fornecedor TEXT,
    ativo_ano_anterior TEXT,
    volume_vendas_ano REAL
);
''')

# Tabela 5: Compra
cursor.execute('''
CREATE TABLE IF NOT EXISTS Compra (
    id_compra TEXT PRIMARY KEY,
    ano_compra INTEGER,
    insercao TEXT,
    modalidade_compra TEXT,
    tipo_compra TEXT,
    cnpj_instituicao TEXT,
    FOREIGN KEY (cnpj_instituicao) REFERENCES Instituicao(cnpj_instituicao)
);
''')

# Tabela Associativa: Item_Compra (Relacionamento M:N)
cursor.execute('''
CREATE TABLE IF NOT EXISTS Item_Compra (
    id_compra TEXT,
    codigo_br TEXT,
    cnpj_fornecedor TEXT,
    qtd_itens_comprados INTEGER,
    preco_unitario REAL,
    preco_total REAL,
    PRIMARY KEY (id_compra, codigo_br, cnpj_fornecedor),
    FOREIGN KEY (id_compra) REFERENCES Compra(id_compra),
    FOREIGN KEY (codigo_br) REFERENCES Produto(codigo_br),
    FOREIGN KEY (cnpj_fornecedor) REFERENCES Fornecedor(cnpj_fornecedor)
);
''')

# ==========================================
# DML - POPULANDO AS TABELAS COM PROCESSO ELT
# ==========================================

print("Populando a tabela Instituicao...")
cursor.execute('''
INSERT OR IGNORE INTO Instituicao (cnpj_instituicao, nome_instituicao, esfera, municipio_instituicao, uf)
SELECT cnpj_instituicao, MAX(nome_instituicao), MAX(esfera), MAX(municipio_instituicao), MAX(uf)
FROM dados_brutos WHERE cnpj_instituicao IS NOT NULL GROUP BY cnpj_instituicao;
''')

print("Populando a tabela Fabricante com cálculos históricos...")
cursor.execute('''
INSERT OR IGNORE INTO Fabricante (cnpj_fabricante, fabricante, ativo_ano_anterior, qtd_produtos_distintos)
SELECT 
    cnpj_fabricante,
    MAX(fabricante),
    CASE WHEN EXISTS (
        SELECT 1 FROM dados_brutos db2 
        WHERE db2.cnpj_fabricante = dados_brutos.cnpj_fabricante AND db2.ano_compra = 2025
    ) THEN 'Sim' ELSE 'Não' END,
    COUNT(DISTINCT codigo_br)
FROM dados_brutos
WHERE cnpj_fabricante IS NOT NULL
GROUP BY cnpj_fabricante;
''')

print("Populando a tabela Produto...")
cursor.execute('''
INSERT OR IGNORE INTO Produto (codigo_br, descricao_catmat, generico, anvisa, capacidade, unidade_medida, unidade_fornecimento_capacidade, cnpj_fabricante)
SELECT codigo_br, MAX(descricao_catmat), MAX(generico), MAX(anvisa), MAX(capacidade), MAX(unidade_medida), MAX(unidade_fornecimento_capacidade), MAX(cnpj_fabricante)
FROM dados_brutos WHERE codigo_br IS NOT NULL GROUP BY codigo_br;
''')

print("Populando a tabela Fornecedor com cálculos históricos...")
cursor.execute('''
INSERT OR IGNORE INTO Fornecedor (cnpj_fornecedor, fornecedor, ativo_ano_anterior, volume_vendas_ano)
SELECT 
    cnpj_fornecedor,
    MAX(fornecedor),
    CASE WHEN EXISTS (
        SELECT 1 FROM dados_brutos db2 
        WHERE db2.cnpj_fornecedor = dados_brutos.cnpj_fornecedor AND db2.ano_compra = 2025
    ) THEN 'Sim' ELSE 'Não' END,
    COALESCE((
        SELECT SUM(db3.preco_total) FROM dados_brutos db3 
        WHERE db3.cnpj_fornecedor = dados_brutos.cnpj_fornecedor AND db3.ano_compra = 2026
    ), 0)
FROM dados_brutos
WHERE cnpj_fornecedor IS NOT NULL
GROUP BY cnpj_fornecedor;
''')

print("Populando a tabela Compra...")
cursor.execute('''
INSERT OR IGNORE INTO Compra (id_compra, ano_compra, insercao, modalidade_compra, tipo_compra, cnpj_instituicao)
SELECT compra, MAX(ano_compra), MAX(insercao), MAX(modalidade_compra), MAX(tipo_compra), MAX(cnpj_instituicao)
FROM dados_brutos WHERE compra IS NOT NULL GROUP BY compra;
''')

print("Populando a tabela transacional Item_Compra (M:N)...")
cursor.execute('''
INSERT OR IGNORE INTO Item_Compra (id_compra, codigo_br, cnpj_fornecedor, qtd_itens_comprados, preco_unitario, preco_total)
SELECT compra, codigo_br, cnpj_fornecedor, qtd_itens_comprados, preco_unitario, preco_total
FROM dados_brutos 
WHERE compra IS NOT NULL AND codigo_br IS NOT NULL AND cnpj_fornecedor IS NOT NULL;
''')

# Reativar validações de integridade referencial
cursor.execute("PRAGMA foreign_keys = ON;")
conn.commit()

print("Pipeline de normalização concluído com sucesso!")

Iniciando a criação do esquema e migração dos dados...
Populando a tabela Instituicao...
Populando a tabela Fabricante com cálculos históricos...
Populando a tabela Produto...
Populando a tabela Fornecedor com cálculos históricos...
Populando a tabela Compra...
Populando a tabela transacional Item_Compra (M:N)...
Pipeline de normalização concluído com sucesso!


In [1]:
import pandas as pd
import sqlite3

# 1. Conectar ao banco de dados relacional
conn = sqlite3.connect('trabalho_pratico_2.db')

# Lista com todas as tabelas estruturadas no modelo 3FN
tabelas = ['Instituicao', 'Fabricante', 'Produto', 'Fornecedor', 'Compra', 'Item_Compra']

print("==================================================")
print("      VERIFICAÇÃO DAS DIMENSÕES E INSTÂNCIAS      ")
print("==================================================")

total_acumulado = 0

for tabela in tabelas:
    # Consultar o total de linhas (tuplas) da tabela atual
    query_count = f"SELECT COUNT(*) as total FROM {tabela};"
    total_linhas = pd.read_sql_query(query_count, conn).iloc[0]['total']
    total_acumulado += total_linhas
    
    # Puxar uma pequena amostra de 3 linhas para conferência dos atributos
    query_amostra = f"SELECT * FROM {tabela} LIMIT 3;"
    df_amostra = pd.read_sql_query(query_amostra, conn)
    
    print(f"\n📌 TABELA: {tabela}")
    print(f"🔹 Total de instâncias (tuplas): {total_linhas}")
    print("🔹 Estrutura e Amostra dos dados:")
    display(df_amostra)
    print("-" * 50)

print(f"\n📊 TOTAL DE INSTÂNCIAS ACUMULADAS NO BANCO: {total_acumulado}")
if total_acumulado >= 10000:
    print("✅ Requisito de tamanho mínimo atendido (> 10.000 tuplas)!")
else:
    print("⚠️ Atenção: O total geral está abaixo de 10.000 tuplas. Verifique a carga de dados.")

      VERIFICAÇÃO DAS DIMENSÕES E INSTÂNCIAS      

📌 TABELA: Instituicao
🔹 Total de instâncias (tuplas): 831
🔹 Estrutura e Amostra dos dados:


,cnpj_instituicao,nome_instituicao,esfera,municipio_instituicao,uf
0,00.136.858/0001-88,CONSORCIO INTERMUNICIPAL DE SAUDE,MUNICIPAL,PATO BRANCO,PR
1,00.204.125/0001-33,SECRETARIA MUNICIPAL DE SAUDE,MUNICIPAL,MACEIO,AL
2,00.333.678/0001-96,CONSORCIO INTERMUNICIPAL DE SAUDE DO SUDOESTE ...,MUNICIPAL,FRANCISCO BELTRAO,PR


--------------------------------------------------

📌 TABELA: Fabricante
🔹 Total de instâncias (tuplas): 2290
🔹 Estrutura e Amostra dos dados:


,cnpj_fabricante,fabricante,ativo_ano_anterior,qtd_produtos_distintos
0,00.008.125/0001-68,MENUCHI ACABAMENTO DE PLASTICOS INDUSTRIA COM ...,Não,6
1,00.008.354/0001-82,BIOSENSOR INDUSTRIA E COMERCIO LTDA,Não,6
2,00.015.955/0001-12,SDI BRASIL INDUSTRIA E COMERCIO LTDA,Sim,24


--------------------------------------------------

📌 TABELA: Produto
🔹 Total de instâncias (tuplas): 12994
🔹 Estrutura e Amostra dos dados:


...

--------------------------------------------------

📌 TABELA: Compra
🔹 Total de instâncias (tuplas): 1887
🔹 Estrutura e Amostra dos dados:


,id_compra,ano_compra,insercao,modalidade_compra,tipo_compra,cnpj_instituicao
0,01/01/2020,2020,19/01/2024,Pregão,ADMINISTRATIVA,43.976.166/0001-50
1,01/01/2021,2021,29/12/2023,Pregão,ADMINISTRATIVA,80.905.706/0001-31
2,01/01/2022,2022,23/01/2024,Pregão,ADMINISTRATIVA,18.478.187/0001-07


--------------------------------------------------

📌 TABELA: Item_Compra
🔹 Total de instâncias (tuplas): 334530
🔹 Estrutura e Amostra dos dados:


,id_compra,codigo_br,cnpj_fornecedor,qtd_itens_comprados,preco_unitario,preco_total
0,01/01/2020,270019,04.949.905/0001-63,9750,4.500,43875.0
1,01/01/2020,243488,06.974.929/0001-06,3,6.900,20.7
2,01/01/2020,267567,01.140.868/0001-50,36000,0.155,5580.0


--------------------------------------------------

📊 TOTAL DE INSTÂNCIAS ACUMULADAS NO BANCO: 356034
✅ Requisito de tamanho mínimo atendido (> 10.000 tuplas)!
